# 11 - Inference with Diac (Text+ASR) Model

**Paper:** "Automatic Restoration of Diacritics for Speech Data Sets" (NAACL 2024)
**Repository:** https://github.com/rufaelfekadu/Diac
**Model:** `rufaelfekadu/diac-transformer-text-asr-tashkeela-clartts-kssa`

## Performance on KSSA:
- **DER:** 10.70% (with case ending) / 7.47% (without)
- **WER:** 36.60% (with case ending) / 21.35% (without)

## Approach:
- Uses **Text + ASR outputs** as input (multimodal)
- Fine-tuned Whisper provides rough diacritized transcripts
- Transformer model combines both inputs for final diacritization

In [ ]:
!pip install -q transformers torch accelerate tqdm librosa soundfile pandas
!pip install -q git+https://github.com/rufaelfekadu/Diac.git

In [ ]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities
from utils_diacritization import (
    normalize_arabic, postprocess_diacritics, compute_metrics,
    is_valid_output, repair_output
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'

for d in [OUTPUT_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

## 1. Clone Diac Repository and Setup Constants

In [ ]:
# Clone Diac repo for constants
DIAC_DIR = PROJECT_DIR / 'Diac'
if not DIAC_DIR.exists():
    !git clone https://github.com/rufaelfekadu/Diac.git {DIAC_DIR}
    print(f"Cloned Diac to {DIAC_DIR}")
else:
    print(f"Diac already exists at {DIAC_DIR}")

# Add to path
sys.path.insert(0, str(DIAC_DIR))

In [ ]:
# Load the Diac model
from diac.models import DiacritizationModule

MODEL_NAME = 'rufaelfekadu/diac-transformer-text-asr-tashkeela-clartts-kssa'
MODEL_KEY = 'diac_text_asr'

print(f"Loading {MODEL_NAME}...")
model = DiacritizationModule.from_pretrained(
    MODEL_NAME,
    tokenizer_constants_path=str(DIAC_DIR / 'constants')
)
model.eval()
if device == "cuda":
    model = model.to(device)
print("Model loaded!")

## 2. Load Whisper for ASR (to provide ASR input)

In [ ]:
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load Whisper for ASR
print("Loading Whisper for ASR...")
whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-small')
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    'openai/whisper-small',
    torch_dtype=torch.float32
).to(device)
whisper_model.eval()
print("Whisper loaded!")

def load_audio(path, sr=16000):
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except:
        return None

@torch.inference_mode()
def whisper_transcribe(audio_path):
    """Transcribe audio using Whisper."""
    audio = load_audio(audio_path)
    if audio is None:
        return None
    
    # Truncate if too long (30 seconds)
    max_length = 30 * 16000
    if len(audio) > max_length:
        audio = audio[:max_length]
    
    input_features = whisper_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)
    
    generated_ids = whisper_model.generate(
        input_features,
        language="ar",
        task="transcribe"
    )
    
    return whisper_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

## 3. Inference Function

In [ ]:
@torch.inference_mode()
def diacritize_with_asr(text: str, audio_path: str = None) -> str:
    """
    Diacritize text using Diac model with optional ASR input.
    
    The model expects:
    - Undiacritized text
    - ASR output (rough transcript from Whisper)
    """
    original_text = text
    text = normalize_arabic(text)
    
    # Get ASR output if audio is available
    asr_output = None
    if audio_path and Path(audio_path).exists():
        asr_output = whisper_transcribe(audio_path)
    
    try:
        # Use Diac model for diacritization
        if asr_output:
            # Text+ASR mode
            result = model.predict_text_with_asr(text, asr_output)
        else:
            # Text-only mode (fallback)
            result = model.predict_text(text)
        
        # Post-process
        result = postprocess_diacritics(result)
        
        # Validate
        if not is_valid_output(result, original_text):
            return original_text
        
        return result
    except Exception as e:
        print(f"Error: {e}")
        return original_text

## 4. Evaluate on Dev Set

In [ ]:
# Load dev data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

# Checkpoint support
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

# Run inference
processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        result = diacritize_with_asr(item['text_undiacritized'], str(audio_path))
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS (Diac Text+ASR)")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab (case endings)]")
print(f"  DER: {metrics['DER']*100:.2f}%")
print(f"  WER: {metrics['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics['SER']*100:.2f}%")

metrics_no_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=True)
print(f"\n[Excluding I'rab (case endings)]")
print(f"  DER: {metrics_no_irab['DER']*100:.2f}%")

## 5. Generate Test Submission

In [ ]:
# Load test data
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

# Run inference
test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        result = diacritize_with_asr(item['text_undiacritized'], str(audio_path))
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")

In [ ]:
# Sync and cleanup
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script), '--with-checkpoints'])

# Final cleanup
print("\nCleaning up GPU memory...")
del model
del whisper_model
del whisper_processor
clear_gpu()
print("Done! GPU memory released.")